In [1]:
!nvidia-smi

Sat Mar  7 06:23:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 1: Setup — Clone repo, install deps, verify GPU
import os, subprocess, sys, shutil, time

t0 = time.time()

# ── Clone repo ──
REPO = "/content/nst"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.run(["git", "clone", "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git", REPO], check=True)
    print("Repo cloned")
else:
    r = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True, cwd=REPO)
    print(f"git pull: {r.stdout.strip()}")

os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

# ── Install dependencies (DO NOT install torch — use Colab's pre-installed CUDA version) ──
print("\nInstalling dependencies (keeping pre-installed PyTorch)...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.3", "datasets==2.21.0",
    "sentencepiece>=0.1.99", "protobuf>=4.0",
    "pyyaml", "scikit-learn", "rank-bm25==0.2.2",
    "accelerate", "peft", "tiktoken",
    "fsspec>=2023.6,<2025", "huggingface_hub>=0.21,<1.0"],
    check=True, capture_output=True, text=True)
print("pip install done")

# Install repo in editable mode
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
    capture_output=True, text=True)

# ── Verify ──
import torch
import transformers
import datasets

print(f"\nPyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM         : {vram / 1e9:.1f} GB")
else:
    print("WARNING: No CUDA GPU! Check Runtime > Change runtime type > T4 GPU")
print(f"\nSetup complete ({time.time()-t0:.0f}s)")

: 

In [4]:
# Cell 2: Build FEVER wiki cache (one-time, ~4 min)
import os, sys, time

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_wiki_cache import WikiCache

cache_path = "data/fever_wiki.db"
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    print(f"Wiki cache already exists: {n} pages, {os.path.getsize(cache_path)/1024/1024:.1f} MB")
    cache.close()
else:
    print("Building wiki cache from HuggingFace FEVER dataset...")
    print("This downloads ~300MB and takes ~4 minutes. One-time only.")
    t0 = time.time()
    import subprocess
    r = subprocess.run([sys.executable, "main.py", "build-fever-wiki-cache"],
                       capture_output=True, text=True, timeout=600)
    print(r.stdout[-2000:] if r.stdout else "(no stdout)")
    if r.returncode != 0:
        print(f"STDERR: {r.stderr[-1000:]}")
    else:
        print(f"\nWiki cache built in {time.time()-t0:.0f}s")

# Verify
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    sample = cache.titles()[:3]
    for t in sample:
        sents = cache.lookup(t)
        print(f"  '{t}': {len(sents)} sentences")
    cache.close()
    print(f"Wiki cache verified: {n} pages")
else:
    print("ERROR: Wiki cache not found!")

Building wiki cache from HuggingFace FEVER dataset...
This downloads ~300MB and takes ~4 minutes. One-time only.

  Built: 14363/14533 pages (170 missing) in 363.6s
  Cache: data/fever_wiki.db (24.2 MB)


Wiki cache built in 365s
  '"Heroes"_-LRB-David_Bowie_album-RRB-': 9 sentences
  ''Til_Death': 4 sentences
  '...More_Unchartered_Heights_of_Disgrace': 13 sentences
Wiki cache verified: 14363 pages


In [5]:
# Cell 3: Smoke Test — 200 examples, 1 epoch (~2 min)
import os, sys, time, logging

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  SMOKE TEST: DeBERTa-v3-base, 200 train, 100 dev, 1 epoch")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results = train_fever_nst("configs/fever_gold_smoke.yaml")
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"SMOKE TEST RESULTS ({elapsed:.0f}s):")
print("=" * 60)
if isinstance(results, dict):
    dev = results.get("dev", {})
    print(f"  dev_accuracy : {dev.get('accuracy', 'N/A')}")
    print(f"  dev_ece      : {dev.get('ece', 'N/A')}")
    print(f"  nan_abort    : {results.get('nan_abort', False)}")
    if results.get("nan_abort", False):
        print("\n SMOKE TEST FAILED -- NaN detected!")
    else:
        print("\n Smoke test passed! Pipeline working correctly.")

  SMOKE TEST: DeBERTa-v3-base, 200 train, 100 dev, 1 epoch



train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 200 examples (105 with evidence text, 95 without)
fever_dataset |   dev: 100 examples (47 with evidence text, 53 without)
fever_dataset |   train hash: 398028c26c5ec3e7
fever_dataset |   dev hash: e3b09c2bfa26d269
train_fever | Using GOLD EVIDENCE mode (Setting A

  FEVER Dataset Statistics

  train: 200 examples
    With gold evidence: 105 (52.5%)
    Label distribution:
      SUPPORTS                129  (64.5%)
      REFUTES                  16  (8.0%)
      NOT ENOUGH INFO          55  (27.5%)
    Split hash: 398028c26c5ec3e7

  dev: 100 examples
    With gold evidence: 47 (47.0%)
    Label distribution:
      SUPPORTS                 46  (46.0%)
      REFUTES                  30  (30.0%)
      NOT ENOUGH INFO          24  (24.0%)
    Split hash: e3b09c2bfa26d269


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

  Epoch 1/1: loss=1.1638 constraint=0.0000 | dev_acc=0.3300 ECE=0.0340

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


  Learned temperature: T = 2.3206

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
train_fever | Report saved to outputs_fever_gold_smoke/report.json


  Label Accuracy (GOLD evidence): 0.3300
  ECE: 0.0340
  Brier: 0.6808
    SUPPORTS: acc=0.0000 (n=46)
    REFUTES: acc=0.3000 (n=30)
    NOT ENOUGH INFO: acc=1.0000 (n=24)

  Training complete in 8.5s
  Best dev accuracy: 0.3300
  Output: outputs_fever_gold_smoke

SMOKE TEST RESULTS (72s):
  dev_accuracy : 0.33
  dev_ece      : 0.033954
  nan_abort    : False

 Smoke test passed! Pipeline working correctly.


In [6]:
# Cell 4: NEURAL BASELINE — Full data, 3 epochs (~45 min on T4)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NEURAL BASELINE (Setting A: Gold Evidence)")
print("  DeBERTa-v3-base, Full FEVER train (~145K), 3 epochs")
print("  batch=16, grad_accum=2 (eff. batch=32)")
print("  Eval: every 500 steps on 2K dev subset")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural = train_fever_nst("configs/fever_gold_neural.yaml")
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"NEURAL BASELINE RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
dev = results_neural.get("dev", {})
dev_test = results_neural.get("dev_test", {})
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  temperature       : {results_neural.get('temperature', 'N/A')}")
print(f"  best_dev_acc      : {results_neural.get('best_dev_acc', 'N/A')}")
print(f"\nPer-label:")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural.json", "w") as f:
    json.dump(results_neural, f, indent=2)
print(f"\nNeural baseline done in {elapsed/60:.1f} min")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NEURAL BASELINE (Setting A: Gold Evidence)
  DeBERTa-v3-base, Full FEVER train (~145K), 3 epochs
  batch=16, grad_accum=2 (eff. batch=32)
  Eval: every 500 steps on 2K dev subset



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Learned temperature: T = 1.1094

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8252
  ECE: 0.0286
  Brier: 0.2445
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

────────────────────────────────────────
  Final evaluation on held-out dev_test
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8255
  ECE: 0.0384
  Brier: 0.2453
    SUPPORTS: acc=0.8788 (n=652)
    REFUTES: acc=0.8364 (n=697)
    NOT ENOUGH INFO: acc=0.7604 (n=651)

  Training complete in 4405.4s
  Best dev accuracy: 0.8430
  Output: outputs_fever_gold_neural

NEURAL BASELINE RESULTS (81.4 min):
  dev accuracy      : 0.8252
  dev ECE           : 0.028639
  dev Brier         : 0.24446
  dev_test accuracy : 0.8255
  dev_test ECE      : 0.03835
  temperature       : 1.1094
  best_dev_acc      : 0.843

Per-label:
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

Neural baseline done in 81.4 min


In [7]:
# Cell 5: NST SOFT CONSTRAINTS — Fixed lambda=0.1 (~45 min on T4)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST SOFT CONSTRAINTS (Setting A: Gold Evidence)")
print("  CE + fixed lambda=0.1 x constraint_loss")
print("  5 constraints: date, number, negation, entity, empty")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_soft = train_fever_nst("configs/fever_gold_nst_soft.yaml")
elapsed = time.time() - t0

dev = results_soft.get("dev", {})
dev_test = results_soft.get("dev_test", {})
print("\n" + "=" * 60)
print(f"NST SOFT RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  final lambda      : {results_soft.get('final_lambda', 'N/A')}")

with open("results_soft.json", "w") as f:
    json.dump(results_soft, f, indent=2)
print(f"\nNST Soft done in {elapsed/60:.1f} min")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NST SOFT CONSTRAINTS (Setting A: Gold Evidence)
  CE + fixed lambda=0.1 x constraint_loss
  5 constraints: date, number, negation, entity, empty



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Step 4500: loss=0.3039 λ=0.1000 | dev_acc=0.8415 ECE=0.0185


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Epoch 1/3: loss=0.5839 constraint=0.4076


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

: 

In [1]:
# Cell 6: NST CEGIS — Counterexample-Guided (~60 min on T4)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST CEGIS (Setting A: Gold Evidence)")
print("  Lagrangian + counterexample-guided outer loop")
print("  Max 5 CEGIS rounds, counterexamples mined from TRAIN only")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_cegis = train_fever_nst("configs/fever_gold_nst_cegis.yaml")
elapsed = time.time() - t0

dev = results_cegis.get("dev", {})
dev_test = results_cegis.get("dev_test", {})
cegis_info = results_cegis.get("cegis", {})
print("\n" + "=" * 60)
print(f"NST CEGIS RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  CEGIS rounds      : {cegis_info.get('total_rounds', 'N/A')}")
print(f"  CEGIS converged   : {cegis_info.get('converged', 'N/A')}")
print(f"  final lambda      : {results_cegis.get('final_lambda', 'N/A')}")

with open("results_cegis.json", "w") as f:
    json.dump(results_cegis, f, indent=2)
print(f"\nNST CEGIS done in {elapsed/60:.1f} min")

FileNotFoundError: [Errno 2] No such file or directory: '/content/nst'

In [ ]:
# Cell 7: NST ECCG (GATED) — Evidence-Conditioned Constraint Gating (~50 min)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST ECCG / GATED (Setting A: Gold Evidence)")
print("  Evidence-Conditioned Constraint Gating (NOVEL)")
print("  Learns when to apply each constraint per-sample")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_gated = train_fever_nst("configs/fever_gold_nst_gated.yaml")
elapsed = time.time() - t0

dev = results_gated.get("dev", {})
dev_test = results_gated.get("dev_test", {})
print("\n" + "=" * 60)
print(f"NST ECCG RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  final lambda      : {results_gated.get('final_lambda', 'N/A')}")

with open("results_gated.json", "w") as f:
    json.dump(results_gated, f, indent=2)
print(f"\nNST ECCG done in {elapsed/60:.1f} min")

In [ ]:
# Cell 8: Results Summary — Comparison Table
import json, os

os.chdir("/content/nst")

print("=" * 70)
print("  FEVER RESULTS — Setting A: Gold Evidence, DeBERTa-v3-base")
print("=" * 70)
print()

# Load all results
results = {}
for name, path in [("Neural", "results_neural.json"),
                    ("Soft", "results_soft.json"),
                    ("CEGIS", "results_cegis.json"),
                    ("ECCG", "results_gated.json")]:
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
    else:
        print(f"  WARNING: {name} not found ({path})")

if not results:
    print("No results found! Run cells 4-7 first.")
else:
    # Print table header
    print(f"  {'Mode':<12} {'Dev Acc':<10} {'ECE':<10} {'Brier':<10} {'DevTest Acc':<12} {'lambda':<8}")
    print("  " + "-" * 68)

    for name, r in results.items():
        dev = r.get("dev", {})
        dt = r.get("dev_test", {})
        lam = r.get("final_lambda", 0)
        dev_acc = dev.get("accuracy", 0)
        dev_ece = dev.get("ece", 0)
        dev_brier = dev.get("brier", 0)
        dt_acc = dt.get("accuracy", 0) if dt else "N/A"
        if isinstance(dt_acc, float):
            dt_str = f"{dt_acc:<12.4f}"
        else:
            dt_str = f"{dt_acc:<12}"
        if isinstance(lam, (int, float)):
            lam_str = f"{lam:<8.4f}"
        else:
            lam_str = f"{str(lam):<8}"
        print(f"  {name:<12} {dev_acc:<10.4f} {dev_ece:<10.4f} {dev_brier:<10.4f} {dt_str} {lam_str}")

    print()
    print("  Key:")
    print("  - Dev Acc: Label accuracy on dev set (tuning split)")
    print("  - DevTest Acc: Label accuracy on held-out 10% (final metric)")
    print("  - ECE: Expected Calibration Error (lower = better)")
    print("  - Brier: Brier score (lower = better)")

    # Find best dev acc
    best_name = max(results, key=lambda n: results[n].get("dev", {}).get("accuracy", 0))
    best_acc = results[best_name].get("dev", {}).get("accuracy", 0)
    print(f"\n  BEST: {best_name} (dev_acc={best_acc:.4f})")

    # Per-label breakdown for best model
    best_dev = results[best_name].get("dev", {})
    print(f"\n  Per-label breakdown ({best_name}):")
    for label, stats in best_dev.get("per_label", {}).items():
        print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

    # Find best calibrated
    best_cal = min(results, key=lambda n: results[n].get("dev", {}).get("ece", 1.0))
    best_ece = results[best_cal].get("dev", {}).get("ece", 0)
    print(f"\n  BEST CALIBRATED: {best_cal} (ECE={best_ece:.4f})")

print("\nAll experiments complete!")
print("Full reports in outputs_fever_gold_*/report.json")
print("\nRemember to stop Colab runtime: Runtime > Disconnect and delete runtime")